# 阶段 1 · 第 1 课：导数与反向传播

大模型是怎么「学习」的？说到底只有一件事：**算出每个参数该往哪个方向调、调多少**。这个「方向和力度」就是**梯度**，算梯度的方法叫**反向传播**。

这一课从最简单的导数开始，最后你会亲手写出一个能自动算梯度的小引擎（仿照 Karpathy 的 micrograd），并验证它和 PyTorch 算出来的一模一样。

**怎么上**：从上往下 `Shift+Enter`。带 `TODO` 的格子先填好再运行。

**提示怎么用**：灰色的「▶ 提示」点开才显示。API 和语法会直接写在格子上方；涉及概念的地方分几级提示，想自己挑战就别点，卡住了一级一级点开。

In [6]:
import math

## 第 1 步 · 导数：输入动一点点，输出动多少

看这个函数：

$$f(x) = 3x^2 - 4x + 5$$

导数回答的问题是：**x 往右挪一点点，f 会跟着变多少？** 变化量除以挪动的距离，就是 f 在这一点的**斜率**。

写成公式（h 取一个很小的数，比如 0.0001）：

$$\text{斜率} \approx \frac{f(x+h) - f(x)}{h}$$

这种「真的挪一点点去量」的算法叫**数值导数**。

In [7]:
def f(x):
    return 3 * x**2 - 4 * x + 5


h = 0.0001
x = 3.0

# TODO 1：用上面的公式，算 f 在 x=3 处的斜率
slope = (f(x+h)-f(x))/h

assert slope is not None, "先完成 TODO 1"
print(f"数值导数:          {slope:.4f}")
print(f"微积分公式 6x - 4: {6 * x - 4}")

数值导数:          14.0003
微积分公式 6x - 4: 14.0


两个数应该几乎一样。**数值导数**是挪一点点去量；**6x − 4** 是用微积分推出来的精确公式。

**想一想**：把上面的 `x` 改成 `2/3` 再跑一次，斜率是多少？这说明了什么？

答：x = 2/3的时候，斜率等于0 ，说明此时向左或者向右移动x，f(x)都没有变化

<details><summary>提示</summary>

斜率是 0，意味着 x 往哪边挪，f 都几乎不变。画一下 f 的图像，这个点在哪？

</details>

<details><summary>答案</summary>

x = 2/3 是这条抛物线的**谷底**，斜率为 0。

训练模型做的就是同一件事：把 loss（模型错了多少）当成函数，沿着让它下降的方向走，一直走到谷底附近。

</details>

## 第 2 步 · 多个输入：每个输入各有一个导数

模型的输出取决于很多个参数。先看三个输入的例子：

$$d = a \times b + c$$

现在要问三个问题：a 动一点点 d 变多少？b 呢？c 呢？下面这格分别挪一下 a、b、c 去量（不用填，直接运行）：

In [8]:
def d_of(a, b, c):
    return a * b + c


a, b, c = 2.0, -3.0, 10.0
h = 0.0001
base = d_of(a, b, c)

print("d 对 a 的导数:", (d_of(a + h, b, c) - base) / h)
print("d 对 b 的导数:", (d_of(a, b + h, c) - base) / h)
print("d 对 c 的导数:", (d_of(a, b, c + h) - base) / h)

d 对 a 的导数: -3.000000000010772
d 对 b 的导数: 2.0000000000042206
d 对 c 的导数: 0.9999999999976694


**找规律**：三个导数分别等于什么？和 a、b、c 的值有什么关系？

三个导数分别是-3，2和1.分别对应着b，a的数值，d对c的导数是1。

<details><summary>提示</summary>

对比一下：d 对 a 的导数是 −3，b 的值也是 −3。

</details>

<details><summary>答案</summary>

- **乘法**：d 对 a 的导数 = **b 的值**，d 对 b 的导数 = **a 的值**。也就是「对方的值」。
- **加法**：d 对 c 的导数 = **1**。c 多 1，d 就多 1，原样传过去。

这两条规则就是下一步引擎的核心。

</details>

## 第 3 步 · 链式法则：一层一层往回传

真实的模型是很多层运算叠起来的。再加一层：

$$L = d \times f, \quad \text{其中 } d = a \times b + c$$

想知道 **L 对 a 的导数**，可以分两段：

1. L 对 d 的导数：乘法规则，是对方的值 f
2. d 对 a 的导数：上一步算过，是 b

**链式法则**：两段**相乘**，L 对 a 的导数 = (L 对 d 的导数) × (d 对 a 的导数)。

意思是：a 动一点 → d 动 b 倍 → L 再动 f 倍，所以总共是 b × f 倍。

从最终输出 L 开始，一层一层往回乘，就能算出每个输入的导数。这就是**反向传播**。

**先猜**：a=2, b=−3, c=10, f=−2 时，L 对 a 的导数是多少？

6

<details><summary>提示 1</summary>

先算 L 对 d 的导数：L = d × f，是对方的值，等于 f = −2。

</details>

<details><summary>提示 2</summary>

再算 d 对 a 的导数：d = a × b + c，等于 b = −3。

</details>

<details><summary>答案</summary>

两段相乘：(−2) × (−3) = **6**。

</details>

## 第 4 步 · 自己搭一个自动求导引擎

**终点预告**：这一步结束时，不管公式长什么样，只要写一句 `backward`，所有梯度就自动算好：

```python
L = (a * b + c) * f
L.backward()
print(a.grad)   # 6.0
```

PyTorch 训练大模型，用的就是同样的机制。

怎么一步步走到那里？每一小步都会先碰到一个**麻烦**，再用一样新东西解决它：

| 小步 | 上一步留下的麻烦 | 这一步的办法 | 新的 Python 写法 |
|---|---|---|---|
| 4.1 | 自己写的类不能用 `+`、`*` | 让 `Value` 支持加法和乘法 | 运算符重载 |
| 4.2 | 算完就忘了自己是怎么来的 | 每个数记住来源和运算 | 无 |
| 4.3 | 还不知道反向传播长什么样 | 你亲手算一遍 | 无 |
| 4.4 | 「手写不行」只是一句空话 | 换个公式，亲眼看手写的代码出错 | 无 |
| 4.5 | 每换一张图都要重写 | 找出规律，把规则写成函数 | 无 |
| 4.6 | 还得自己记住每个节点是怎么算出来的 | 节点被算出来时，就把规则存好 | 闭包 |
| 4.7 | 规则放在节点外面，绕了一层 | 把规则直接写进节点，改用 `+=` | 无 |
| 4.8 | 调用顺序还得自己写 | 让程序自动排顺序 | 递归 |
| 4.9 | 代码散在各个格子里 | 组装完整引擎，存成文件 | `%%writefile` |

每小步都会把 `Value` 类重新写一遍，新加的行标了 `# ← 新`，只看这几行就行。

### 4.1 · 一个会做加法和乘法的数

先做一个最简单的 `Value`：只存一个数 `data`，能用 `+` 和 `*`。

**语法小抄 · 运算符重载**：Python 里写 `a + b`，实际执行的是 `a.__add__(b)`；写 `a * b`，执行的是 `a.__mul__(b)`。所以只要在类里定义了 `__add__` 和 `__mul__`，自己写的类也能用 `+` 和 `*`。`__repr__` 决定 `print` 的时候显示成什么样。

In [9]:
class Value:
    def __init__(self, data):
        self.data = data

    def __repr__(self):
        return f"Value({self.data})"

    def __add__(self, other):
        return Value(self.data + other.data)

    def __mul__(self, other):
        return Value(self.data * other.data)


a = Value(2.0)
b = Value(-3.0)
print(a + b)         # Python 实际执行的是 a.__add__(b)
print(a.__mul__(b))  # 和 a * b 完全一样

Value(-1.0)
Value(-6.0)


### 4.2 · 让每个数记住自己是怎么来的

4.1 的 `a * b` 算出 −6 之后，就不记得自己是 a 和 b 乘出来的了。可是反向传播要从 L 一路往回找，每个数都得知道**自己是由谁、用什么运算算出来的**。

所以加两个属性：`_prev` 存算出它的那几个数，`_op` 存用的什么运算。另外加个 `label` 起名字，方便打印。

把第 3 步的式子拆开，给中间的 `a * b` 也起个名字叫 e：

```
a ──┐
    (*)── e ──┐
b ──┘         (+)── d ──┐
          c ──┘         (*)── L
                    f ──┘
```

所有数这样连起来，就是一张**计算图**。

In [14]:
class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self._prev = _prev  # ← 新：算出它的那几个数
        self._op = _op      # ← 新：用的什么运算
        self.label = label  # ← 新：名字，方便打印

    def __repr__(self):
        return f"Value({self.label}={self.data})"

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), "+")  # ← 新：记下来源和运算

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), "*")  # ← 新：记下来源和运算


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
print(e)
print(e._prev)
print(e.label)
d = e + c
d.label = "d"
L = d * f
L.label = "L"

# 从 L 开始，顺着 _prev 一层一层往回找
print(L, "是用", L._op, "算出来的，来自", L._prev)
print(d, "是用", d._op, "算出来的，来自", d._prev)
print(e, "是用", e._op, "算出来的，来自", e._prev)

Value(e=-6.0)
(Value(a=2.0), Value(b=-3.0))
e
Value(L=-8.0) 是用 * 算出来的，来自 (Value(d=4.0), Value(f=-2.0))
Value(d=4.0) 是用 + 算出来的，来自 (Value(e=-6.0), Value(c=10.0))
Value(e=-6.0) 是用 * 算出来的，来自 (Value(a=2.0), Value(b=-3.0))


### 4.3 · 亲手做一遍反向传播

再加一个属性 `grad`，存「L 对这个数的导数」，一开始是 0。然后**你自己**从 L 开始往回，一个节点一个节点地填梯度。只用第 2 步找到的两条规则：

- **加法**：梯度原样传给两个输入
- **乘法**：传给一个输入的梯度 = **对方的值 × 结果的梯度**（「× 结果的梯度」就是链式法则）

顺序很重要：先有 d 的梯度，才能往 e 和 c 传；先有 e 的梯度，才能往 a 和 b 传。前面几行已经写好了，你来写最后一层。

<details><summary>提示 1</summary>

照着 `L = d * f` 那两行的样子写。那里结果是 L、两个输入是 d 和 f；这里结果是 e、两个输入是 a 和 b。

</details>

<details><summary>答案</summary>

```python
a.grad = b.data * e.grad
b.grad = a.data * e.grad
```

</details>

In [21]:
class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self.grad = 0.0  # ← 新：L 对这个数的导数，先设为 0
        self._prev = _prev
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data}, grad={self.grad})"  # ← 新：也打印 grad

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), "*")


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"

L.grad = 1.0  # L 对自己的导数是 1

# L = d * f 是乘法：传给 d 的是「f 的值 × L 的梯度」，传给 f 的是「d 的值 × L 的梯度」
d.grad = f.data * L.grad
f.grad = d.data * L.grad

# d = e + c 是加法：原样传
e.grad = d.grad
c.grad = d.grad

# TODO 2：e = a * b 是乘法，照上面 L = d * f 那两行的写法，算出 a.grad 和 b.grad
a.grad = d.grad * 1 * b.data
b.grad = d.grad * 1* a.data

assert a.grad != 0 and b.grad != 0, "先完成 TODO 2"
for v in [L, d, f, e, c, a, b]:
    print(v)

Value(L=-8.0, grad=1.0)
Value(d=4.0, grad=-2.0)
Value(f=-2.0, grad=4.0)
Value(e=-6.0, grad=-2.0)
Value(c=10.0, grad=-2.0)
Value(a=2.0, grad=6.0)
Value(b=-3.0, grad=-4.0)


a 的梯度应该是 6，和你在第 3 步猜的一样。**这就是反向传播**，只不过是你一行一行手写的。

手写好像也不难，几行就写完了。那为什么非要让程序来做？下一小步换个公式试试。

### 4.4 · 换个公式试试

模型是经常改的。现在在 L 后面再乘一个 g = 3，最终输出变成 M：

```
a ──┐
    (*)── e ──┐
b ──┘         (+)── d ──┐
          c ──┘         (*)── L ──┐
                    f ──┘         (*)── M
                              g ──┘
```

现在要算的是「**M** 对每个数的导数」，起点是 `M.grad = 1`。

下面这格把 4.3 手写的那几行**原样照抄**过来，只在最前面加了一句 `M.grad = 1.0`。最后用第 1 步「挪一点点去量」的办法逐个检查，对的打 ✓，错的打 ✗。

`check_M` 是检查工具，不用读懂它的写法，会用就行：它把你算出的梯度和量出来的梯度并排打印。直接运行：

In [22]:
def M_of(a, b, c, f, g):  # M 的公式，给「挪一点点去量」用
    return (a * b + c) * f * g


def check_M():
    vals = {"a": 2.0, "b": -3.0, "c": 10.0, "f": -2.0, "g": 3.0}
    nodes = {"a": a, "b": b, "c": c, "f": f, "g": g}
    h = 1e-6
    base = M_of(**vals)
    for name in vals:
        nudged = dict(vals)  # 复制一份
        nudged[name] += h    # 只挪这一个数
        measured = (M_of(**nudged) - base) / h
        ok = "✓" if abs(nodes[name].grad - measured) < 1e-3 else "✗"
        print(f"dM/d{name}   算出: {nodes[name].grad:7.2f}   挪一点去量: {measured:7.2f}   {ok}")


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

M.grad = 1.0  # 起点换成了 M

# ↓ 下面原样照抄 4.3
L.grad = 1.0
d.grad = f.data * L.grad
f.grad = d.data * L.grad
e.grad = d.grad
c.grad = d.grad
a.grad = b.data * e.grad
b.grad = a.data * e.grad

check_M()

dM/da   算出:    6.00   挪一点去量:   18.00   ✗
dM/db   算出:   -4.00   挪一点去量:  -12.00   ✗
dM/dc   算出:   -2.00   挪一点去量:   -6.00   ✗
dM/df   算出:    4.00   挪一点去量:   12.00   ✗
dM/dg   算出:    0.00   挪一点去量:   -8.00   ✗


**全错了，而且没有任何报错。** 程序照常跑完，只是给出了错的数。为什么？

因为L.grad是错误的，不是1，M对L的导数的g，也就是，L.grad = g.data

<details><summary>提示 1</summary>

照抄的第一行是 `L.grad = 1.0`。现在终点是 M，L 对 M 的导数还是 1 吗？

</details>

<details><summary>提示 2</summary>

M = L × g，乘法规则：L 的梯度 = 对方的值 × 结果的梯度 = g × 1 = 3。

</details>

<details><summary>答案</summary>

L 已经不是终点了，`L.grad` 应该是 3，照抄的代码却写成了 1。后面每一行都用到了 `L.grad`，所以一路错下去。g 是新加的数，照抄的代码里根本没有它，梯度停在 0。

多加一层，手写的反向传播就要跟着改；忘了改，也不会报错。

</details>

**TODO 3**：改对它。M = L × g 是乘法，照 4.3 里 `L = d * f` 那两行的写法，写出 `L.grad` 和 `g.grad`。下面的 6 行不用动。

<details><summary>提示 1</summary>

`L = d * f` 那两行里，结果是 L、两个输入是 d 和 f；这里结果是 M、两个输入是 L 和 g。

</details>

<details><summary>答案</summary>

```python
L.grad = g.data * M.grad
g.grad = L.data * M.grad
```

</details>

In [23]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

M.grad = 1.0

# TODO 3：M = L * g 是乘法，写出 L.grad 和 g.grad
L.grad = g.data  # ← 这一行不对了，改掉
g.grad = L.data

# ↓ 下面 6 行和 4.3 一样，不用动
d.grad = f.data * L.grad
f.grad = d.data * L.grad
e.grad = d.grad
c.grad = d.grad
a.grad = b.data * e.grad
b.grad = a.data * e.grad

assert g.grad != 0, "先完成 TODO 3"
check_M()

dM/da   算出:   18.00   挪一点去量:   18.00   ✓
dM/db   算出:  -12.00   挪一点去量:  -12.00   ✓
dM/dc   算出:   -6.00   挪一点去量:   -6.00   ✓
dM/df   算出:   12.00   挪一点去量:   12.00   ✓
dM/dg   算出:   -8.00   挪一点去量:   -8.00   ✓


全是 ✓ 就对了。现在你亲身体会到了手写的麻烦：

- 模型每改一次，就得有人重新手写对应的那几行。
- 写漏、写错不会报错，只会算出错的数，刚才你亲眼看到了。
- 真实的大模型有几十亿个节点，结构还经常改，靠人写不可能。

所以要让程序自己写这些行。先看看这些行有没有规律。

### 4.5 · 找规律：把规则写成函数

把你刚才写的所有行，按「哪个节点、什么运算」分组排一下：

```python
# 乘法：M = L * g
L.grad = g.data * M.grad
g.grad = L.data * M.grad

# 乘法：L = d * f
d.grad = f.data * L.grad
f.grad = d.data * L.grad

# 加法：d = e + c
e.grad = d.grad
c.grad = d.grad

# 乘法：e = a * b
a.grad = b.data * e.grad
b.grad = a.data * e.grad
```

**想一想**：三组乘法长得有什么共同点？每一行怎么写，取决于什么？

乘法的话，这个数的梯度等于它的另外一个乘数的数值乘以积的梯度
也就是输入1.grad = 输入2.data * 结果.grad
确实，就是这一行怎么写，只看这个阶段的运算逻辑，只用到了它的两个输入和结果，与该节点所处的位置，在写法上没有关心。

<details><summary>提示 1</summary>

把每组里的名字换成「结果」「输入 1」「输入 2」再看。

</details>

<details><summary>提示 2</summary>

每组只用到三样东西：两个输入、一个结果。这个结果再往上还有几层，和这两行怎么写有关系吗？

</details>

<details><summary>答案</summary>

三组乘法是**同一个模板**：

```
输入1.grad = 输入2.data × 结果.grad
输入2.grad = 输入1.data × 结果.grad
```

每一行怎么写，**只看这个节点是什么运算**（加法还是乘法），只用到它自己的两个输入和结果。它在图里的哪个位置、上面还有几层，都没关系。

</details>

**两种写法的区别**：4.3 的 TODO 2 有两种写法，算出来都是 6：

```python
a.grad = d.grad * 1 * b.data   # 写法一：从 d 一路乘下来
a.grad = b.data * e.grad       # 写法二：只看正上方的结果 e
```

写法一要知道 e 上面是 d、中间是加法，换一张图就得重写。写法二只看 e 自己，因为 `e.grad` 里已经存好了「上面所有层乘起来」的结果。

**只看正上方一层，才能变成通用的规则。** 这就是链式法则在代码里的样子：每个节点只管「局部导数 × 结果的梯度」，上面的事交给上面的节点。

既然规则只需要「两个输入 + 一个结果」这三样，就能写成一个普通函数，把这三样当参数传进去。加法的已经写好了。

**TODO 4**：照着 `add_backward` 写 `mul_backward`。

<details><summary>提示 1</summary>

照 4.3 里 `L = d * f` 那两行写：结果 L 换成 `out`，两个输入 d、f 换成 `x`、`y`。

</details>

<details><summary>答案</summary>

```python
x.grad = y.data * out.grad
y.grad = x.data * out.grad
```

</details>

In [26]:
def add_backward(x, y, out):  # out = x + y
    x.grad = out.grad
    y.grad = out.grad


def mul_backward(x, y, out):  # out = x * y
    # TODO 4：照 4.3 里 L = d * f 那两行写，结果叫 out，两个输入叫 x 和 y
    x.grad = y.data * out.grad
    y.grad = x.data * out.grad


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

M.grad = 1.0
mul_backward(L, g, M)  # M = L * g
mul_backward(d, f, L)  # L = d * f
add_backward(e, c, d)  # d = e + c
mul_backward(a, b, e)  # e = a * b

assert a.grad != 0, "先完成 TODO 4"
check_M()

dM/da   算出:   18.00   挪一点去量:   18.00   ✓
dM/db   算出:  -12.00   挪一点去量:  -12.00   ✓
dM/dc   算出:   -6.00   挪一点去量:   -6.00   ✓
dM/df   算出:   12.00   挪一点去量:   12.00   ✓
dM/dg   算出:   -8.00   挪一点去量:   -8.00   ✓


8 行手写，变成了 4 行调用。每一行只需要说清楚：这个节点是**什么运算**、**输入是谁**。

**还剩的麻烦**：写 `mul_backward(d, f, L)` 这一行，你得知道「L 是 d 乘 f 算出来的」。图一大，谁记得住？

其实有一个地方一定知道：**做乘法的那一刻**。

### 4.6 · 让节点自己记住规则

每次写 `d * f`，Python 都会执行 `d.__mul__(f)`。在 `__mul__` 里面，`self` 就是 d，`other` 就是 f，`out` 就是算出来的 L。也就是说，`mul_backward(self, other, out)` 需要的三样东西，**在 `__mul__` 里全都有**。

那做完运算马上调用规则，行不行？`__mul__` 里调用 `mul_backward`，`__add__` 里调用 `add_backward`。每调用一次，就把那一刻的数打印出来。先试试，直接运行：

In [27]:
class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._prev = _prev
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")
        add_backward(self, other, out)  # ← 新：做完加法，马上传梯度？
        # 下面三行只是打印出来看
        print(f"做 {self.label} + {other.label}，这时结果的 grad = {out.grad}")
        print(f"    {self.label}.grad = 结果.grad = {self.grad}")
        print(f"    {other.label}.grad = 结果.grad = {other.grad}")
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")
        mul_backward(self, other, out)  # ← 新：做完乘法，马上传梯度？
        # 下面三行只是打印出来看
        print(f"做 {self.label} * {other.label}，这时结果的 grad = {out.grad}")
        print(f"    {self.label}.grad = {other.label}.data × 结果.grad = {other.data} × {out.grad} = {self.grad}")
        print(f"    {other.label}.grad = {self.label}.data × 结果.grad = {self.data} × {out.grad} = {other.grad}")
        return out


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

print("四次运算都做完了，现在才执行 M.grad = 1.0")
M.grad = 1.0
for v in [M, L, g, d, f, e, c, a, b]:
    print(v)

做 a * b，这时结果的 grad = 0.0
    a.grad = b.data × 结果.grad = -3.0 × 0.0 = -0.0
    b.grad = a.data × 结果.grad = 2.0 × 0.0 = 0.0
做 e + c，这时结果的 grad = 0.0
    e.grad = 结果.grad = 0.0
    c.grad = 结果.grad = 0.0
做 d * f，这时结果的 grad = 0.0
    d.grad = f.data × 结果.grad = -2.0 × 0.0 = -0.0
    f.grad = d.data × 结果.grad = 4.0 × 0.0 = 0.0
做 L * g，这时结果的 grad = 0.0
    L.grad = g.data × 结果.grad = 3.0 × 0.0 = 0.0
    g.grad = L.data × 结果.grad = -8.0 × 0.0 = -0.0
四次运算都做完了，现在才执行 M.grad = 1.0
Value(M=-24.0, grad=1.0)
Value(L=-8.0, grad=0.0)
Value(g=3.0, grad=-0.0)
Value(d=4.0, grad=-0.0)
Value(f=-2.0, grad=0.0)
Value(e=-6.0, grad=0.0)
Value(c=10.0, grad=0.0)
Value(a=2.0, grad=-0.0)
Value(b=-3.0, grad=0.0)


除了 M，梯度全是 0（显示成 `-0.0` 的也是 0，这是小数在电脑里的小怪癖，不用管）。打印里写「结果」，是因为这时结果还没有名字：`e.label = "e"` 要到下一行才执行。

**一行一行看发生了什么。** Python 从上往下执行，每一行只执行一次，规则也只在那一刻跑一次：

| 执行到 | 这时 M 算出来了吗 | 结果的 grad | 规则当场算出 |
|---|---|---|---|
| `e = a * b` | 还没有 | e.grad = 0 | a.grad = -3 × 0 = 0，b.grad = 2 × 0 = 0 |
| `d = e + c` | 还没有 | d.grad = 0 | e.grad = 0，c.grad = 0 |
| `L = d * f` | 还没有 | L.grad = 0 | d.grad = -2 × 0 = 0，f.grad = 4 × 0 = 0 |
| `M = L * g` | 刚算出来 | M.grad = 0 | L.grad = 3 × 0 = 0，g.grad = -8 × 0 = 0 |
| `M.grad = 1.0` | 有了 | M.grad = 1 | 没有规则再跑了 |

第一行就卡住了：`e.grad` 的意思是「M 对 e 的导数」，可是算 e 的时候 M 还不存在，没法知道。`e.grad` 只能是一开始的 0，规则拿 0 去乘，乘出来就是 0。后面每一行都一样。

那最后 `M.grad = 1.0` 了，前面的梯度为什么不跟着变？因为 `a.grad = b.data * e.grad` 是**当场算出一个数存进去**，存的不是公式。之后 `e.grad` 再变，`a.grad` 也不会自己重算。看一个小例子，直接运行：

In [28]:
x = 2
y = x * 10  # 当场算出 20，存进 y
x = 5
print(y)  # 是 20 还是 50？我认为是20 ，因为y已经固定了

20


打印的是 20。Excel 里的公式格会跟着改，Python 的赋值不会。

所以执行 `M.grad = 1.0` 时，前面四次规则早就拿 0 算完了，之后也没有代码再把它们跑一遍，梯度就一直停在 0。

再对比 4.5 那次为什么对了：

| | 规则什么时候跑 | 顺序 |
|---|---|---|
| 4.5 | `M.grad = 1.0` **之后** | M → L → d → e，从结果往回传 |
| 这次 | `M.grad = 1.0` **之前**，每做完一次运算就跑 | e → d → L → M，和前向同一个方向 |

规则本身没错，错在**跑得太早**。

矛盾就在这里：

| 时刻 | 知道输入和结果是谁 | 知道 `out.grad` |
|---|---|---|
| 做乘法时（在 `__mul__` 里） | ✓ | ✗ 还是 0 |
| 反向传播时 | ✗ 早就离开 `__mul__` 了 | ✓ |

办法：在 `__mul__` 里**先不调用**，而是把 `mul_backward(self, other, out)` 这句话，连同三个参数一起**打包存起来**。等反向传播时再拿出来执行。

**语法小抄 · 闭包**：在函数里面再定义一个函数，里面的函数能用外面函数的变量，而且外面的函数执行完之后，它**仍然记得**这些变量：

```python
def make_greeter(name):
    def greet():
        print("你好,", name)  # 用到了外面的 name
    return greet

g = make_greeter("小明")  # make_greeter 已经执行完了
g()                        # 打印「你好, 小明」：greet 还记得 name
```

在这里，闭包就是一个「**参数先记住、以后再执行**」的函数调用。下面 `__mul__` 里的 `_backward` 记住了 `self`、`other`、`out`，以后调用它，就等于执行当时那句 `mul_backward(self, other, out)`。

另外注意：**函数也能存进属性**。`out._backward = _backward` 只是把函数存起来，没有调用；写成 `out._backward()`，带括号，才是调用。

这一格不用填，直接运行：

In [29]:
def do_nothing():
    pass


class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = do_nothing  # ← 新：默认什么都不做（a、b 这种最初的输入，没有可以往回传的）
        self._prev = _prev
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():  # ← 新：把这句话打包，先不执行
            add_backward(self, other, out)

        out._backward = _backward  # ← 新：存进结果节点
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():  # ← 新：把这句话打包，先不执行
            mul_backward(self, other, out)

        out._backward = _backward  # ← 新：存进结果节点
        return out


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

M.grad = 1.0
M._backward()  # 执行 M 存好的那句：mul_backward(L, g, M)
L._backward()  # mul_backward(d, f, L)
d._backward()  # add_backward(e, c, d)
e._backward()  # mul_backward(a, b, e)
check_M()

dM/da   算出:   18.00   挪一点去量:   18.00   ✓
dM/db   算出:  -12.00   挪一点去量:  -12.00   ✓
dM/dc   算出:   -6.00   挪一点去量:   -6.00   ✓
dM/df   算出:   12.00   挪一点去量:   12.00   ✓
dM/dg   算出:   -8.00   挪一点去量:   -8.00   ✓


**想一想**：执行 `M._backward()` 时，里面的 `self`、`other`、`out` 分别是谁？是什么时候记住的？

self是L，other是g，out是M，是在做M = L*g的时候记住的，就相当于运行了L.__mul__(g)

<details><summary>答案</summary>

`self` 是 L，`other` 是 g，`out` 是 M。是在执行 `M = L * g` 那一刻，Python 运行 `L.__mul__(g)` 时记住的。

</details>

和 4.5 比一比：

```python
mul_backward(L, g, M)  # 4.5：得你告诉它输入是 L 和 g
M._backward()          # 4.6：M 自己记得
```

**还剩两个麻烦**：

1. 规则还放在节点外面的 `mul_backward` 里，`_backward` 只是转一手，绕了一层。
2. 调用顺序 `M → L → d → e` 还得自己写。

先解决第一个。

### 4.7 · 把规则直接写进节点

`_backward` 里只有一句 `mul_backward(self, other, out)`，那就把 `mul_backward` 的两行直接搬进来，名字 `x`、`y` 换成 `self`、`other`。micrograd 和 PyTorch 都是这么写的。加法的已经搬好了。

同时把 `=` 换成 `+=`，往梯度上**加**：因为一个数可能被用了好几次，每条路传回来的梯度要加在一起。作业简单题会专门考这一点。

**TODO 5**：写乘法的 `_backward`。

<details><summary>提示 1</summary>

照 4.5 的 `mul_backward` 写：`x` 换成 `self`，`y` 换成 `other`。

</details>

<details><summary>提示 2</summary>

记得把 `=` 换成 `+=`，格式和上面 `__add__` 的 `_backward` 一样。

</details>

<details><summary>答案</summary>

```python
self.grad += other.data * out.grad
other.grad += self.data * out.grad
```

</details>

In [30]:
def do_nothing():
    pass


class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = do_nothing
        self._prev = _prev
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad   # ← 新：规则直接写在这里，改用 +=
            other.grad += out.grad  # ← 新

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # TODO 5：照 4.5 的 mul_backward 写，x、y 换成 self、other，= 换成 +=
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

同一个 `Value`，先算 4.3 的图，再算 4.4 的图。`Value` 一行都不用改：

In [31]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"

L.grad = 1.0
L._backward()
d._backward()
e._backward()
assert a.grad != 0, "a.grad 还是 0：检查 TODO 5"
print("4.3 的图：a.grad =", a.grad, "（应该是 6）")
print()

print("4.4 的图：")

a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

M.grad = 1.0
M._backward()
L._backward()
d._backward()
e._backward()
check_M()

4.3 的图：a.grad = 6.0 （应该是 6）

4.4 的图：
dM/da   算出:   18.00   挪一点去量:   18.00   ✓
dM/db   算出:  -12.00   挪一点去量:  -12.00   ✓
dM/dc   算出:   -6.00   挪一点去量:   -6.00   ✓
dM/df   算出:   12.00   挪一点去量:   12.00   ✓
dM/dg   算出:   -8.00   挪一点去量:   -8.00   ✓


**这就是回报**：换公式，不用再改反向传播的代码。每个节点在被算出来时，自己就带上了「怎么往回传」。

**还剩的麻烦**：`M._backward()`、`L._backward()`……这串调用顺序还得自己写。图一换，这串也得跟着换。

### 4.8 · 让程序自动排好顺序

最后一件手工活：按 `M → L → d → e` 的顺序调用。顺序不能乱：一个节点必须**等所有用到它的节点都把梯度传过来**，才能接着往下传。比如必须先调 `d._backward()` 让 e 拿到梯度，再调 `e._backward()`。

自动排顺序的办法叫**拓扑排序**：从终点出发，顺着 `_prev` 往回走，保证**每个节点都排在它的所有来源后面**。排出来是「从输入到输出」的顺序，倒过来就是反向传播要的顺序。

**语法小抄 · 递归**：函数调用自己。下面的 `build(v)` 先对 v 的每个来源调用 `build`，等来源都排好了，再把 v 自己放进列表。`visited` 是一个集合（set），记下已经处理过的节点，同一个节点不会被放两次。

直接运行，不用填：

In [32]:
def backward(root):
    topo = []
    visited = set()

    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)  # 先把它的来源都排好
            topo.append(v)    # 再排它自己

    build(root)
    print("拓扑顺序（从输入到输出）:", [v.label for v in topo])

    root.grad = 1.0
    for v in reversed(topo):  # 倒过来，从终点开始
        v._backward()


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"
g = Value(3.0, label="g")
M = L * g
M.label = "M"

backward(M)  # 一行搞定，不用手动排顺序
check_M()

拓扑顺序（从输入到输出）: ['a', 'b', 'e', 'c', 'd', 'f', 'L', 'g', 'M']
dM/da   算出:   18.00   挪一点去量:   18.00   ✓
dM/db   算出:  -12.00   挪一点去量:  -12.00   ✓
dM/dc   算出:   -6.00   挪一点去量:   -6.00   ✓
dM/df   算出:   12.00   挪一点去量:   12.00   ✓
dM/dg   算出:   -8.00   挪一点去量:   -8.00   ✓


到这里，手工活全没了：写公式 → 调一次 `backward` → 所有梯度自动算好。

### 4.9 · 组装完整引擎，存成文件

把 4.8 的 `backward` 放进类里当方法，再补几种运算，就是完整的引擎了。新补的运算都是**同一个模式**：算出结果 → 写一个 `_backward` → 存起来。

| 运算 | 局部导数 |
|---|---|
| `a ** k`（乘方） | k · aᵏ⁻¹ |
| `a.tanh()` | 1 − tanh² |
| `-a`、`a - b` | 用「乘以 −1」和加法拼出来，不用单独写 `_backward` |

还有两处小改动：

- 如果另一边是普通数字（比如 `a + 1`），先把它包成 `Value`。
- `__radd__`、`__rmul__` 处理数字在左边的情况，比如 `2 * a`：Python 发现数字 2 不会乘 `Value`，就会改调 `a.__rmul__(2)`。

**语法小抄 · `%%writefile`**：Jupyter 的魔法命令，写在格子第一行。运行时**不执行**代码，而是把代码存成同目录下的文件 `micrograd.py`。第 2 课和作业都会用 `from micrograd import Value` 载入你这里存下的引擎。

**这一格不用填**，乘法的 `_backward` 已经照你 4.7 写的填好了，直接运行。

In [33]:
%%writefile micrograd.py
import math


def do_nothing():
    pass


class Value:
    def __init__(self, data, _prev=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = do_nothing
        self._prev = _prev
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label or 'data'}={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)  # 数字先包成 Value
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "只支持常数次方"
        out = Value(self.data**k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):  # -a
        return self * -1

    def __sub__(self, other):  # a - b
        return self + (-other)

    def __rsub__(self, other):  # 数字 - a
        return other + (-self)

    __radd__ = __add__  # 数字 + a
    __rmul__ = __mul__  # 数字 * a

    def backward(self):  # 就是 4.8 的 backward，搬进类里
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

Writing micrograd.py


上一格只是存了文件。下面这格把它载入进来。`importlib.reload` 的作用是：改了上一格重新运行后，这里能载入最新版本。所以以后改了上一格，**两格都要重新运行**。

In [34]:
import importlib

import micrograd

importlib.reload(micrograd)
from micrograd import Value

## 第 5 步 · 用完整引擎再算一遍

这次用存好的引擎，`L.backward()` 一键算完。不用拆出 e 了，直接写 `a * b + c`。结果应该和 4.3 你手算的一样：

In [35]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
f = Value(-2.0, label="f")
d = a * b + c
d.label = "d"
L = d * f
L.label = "L"

L.backward()

assert a.grad != 0, "a.grad 还是 0：检查 4.9 那两格有没有运行"
for v in [L, d, a, b, c, f]:
    print(v)

Value(L=-8.0000, grad=1.0000)
Value(d=4.0000, grad=-2.0000)
Value(a=2.0000, grad=6.0000)
Value(b=-3.0000, grad=-4.0000)
Value(c=10.0000, grad=-2.0000)
Value(f=-2.0000, grad=4.0000)


再用第 1 步的「挪一点点去量」验证一遍，两边应该一致：

In [36]:
def L_of(a, b, c, f):
    return (a * b + c) * f


vals = {"a": 2.0, "b": -3.0, "c": 10.0, "f": -2.0}
h = 1e-6
base = L_of(**vals)
for name in vals:
    nudged = dict(vals)
    nudged[name] += h
    print(f"数值导数 dL/d{name}: {(L_of(**nudged) - base) / h:8.4f}")

数值导数 dL/da:   6.0000
数值导数 dL/db:  -4.0000
数值导数 dL/dc:  -2.0000
数值导数 dL/df:   4.0000


## 第 6 步 · 和 PyTorch 对答案

PyTorch 的自动求导（autograd）做的就是你刚写的这件事，只不过它的每个节点是一整个张量，而且能在 GPU 上跑。

**要用到的 API**：
- `torch.tensor(2.0, dtype=torch.float64, requires_grad=True)`：`requires_grad=True` 告诉 PyTorch「这个数的梯度我要」。用 float64 是为了和我们的 Python 小数精度一致。
- `结果.backward()`：反向传播，和我们的一样。
- `张量.grad.item()`：取出梯度，变成普通 Python 数字。

直接运行：

In [37]:
import torch

ta = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
tb = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
tc = torch.tensor(10.0, dtype=torch.float64, requires_grad=True)
tf = torch.tensor(-2.0, dtype=torch.float64, requires_grad=True)

tL = (ta * tb + tc) * tf
tL.backward()

for name, ours, theirs in [("a", a, ta), ("b", b, tb), ("c", c, tc), ("f", f, tf)]:
    print(f"d L/d{name}   我们: {ours.grad:6.2f}   PyTorch: {theirs.grad.item():6.2f}")

d L/da   我们:   6.00   PyTorch:   6.00
d L/db   我们:  -4.00   PyTorch:  -4.00
d L/dc   我们:  -2.00   PyTorch:  -2.00
d L/df   我们:   4.00   PyTorch:   4.00


## 跑完想一想

- 4.8 排出来的拓扑顺序里，c 排在 e 前面还是后面，会影响结果吗？只要满足什么条件，顺序就是对的？
- 你写的引擎每个数都是一个 Python 对象。如果模型有几十亿个参数，会遇到什么问题？（作业困难题会展开）

## 本课新词

| 词 | 意思 |
|---|---|
| 导数 / 斜率 | 输入动一点点，输出变多少倍 |
| 梯度（gradient） | 输出对每个输入的导数，合在一起 |
| 链式法则 | 多层运算的导数 = 每一层的局部导数相乘 |
| 反向传播（backprop） | 从输出开始往回，用链式法则算出所有梯度 |
| 计算图 | 记录「谁由谁、用什么运算算出来」的结构，`_prev` 和 `_op` 存的就是它 |
| autograd | PyTorch 里自动做反向传播的那部分 |

下一课：用这个引擎搭一个小神经网络，让它自己学会分类。